# Lucia 0.8B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/lucia_0.8b.ipynb)

Lucia at Qwen3.5-0.8B, the smallest of the family, which answers without thinking by default (every Lucia is trained and read with thinking off, so the small one is taught the task's shape the same way). Two LoRA arms: the recipe verbatim (rank 32, alpha 64, two epochs, lr 2e-4) and a set tweaked for a small model (three epochs, lr 3e-4, the rest the same); each is screened and read on the two held-out sets, and the release takes the better of the two by those reads. The queue then holds the full fine-tune of the same recipe (rank 0: the language model's weights train in fp32 under bf16 autocast, the vision tower frozen, plain Adam at a full fine-tune's rate), screened and read the same way; it ships as a whole model under `full/<model>/<stamp>_<tag>/final`. The card works through `runs/queue_lucia_0.8b.json` in the project repo ([AbstractPhil/qwen-symbolic-thinking-image-training](https://huggingface.co/AbstractPhil/qwen-symbolic-thinking-image-training)); each run leaves `runs/done/<name>.json`, and the cell ends after 45 idle minutes. The base model is [Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B).

Runtime: a GPU (the RTX PRO 6000 or an A100; the small bases fit smaller cards at batch 8) and the `HF_TOKEN` secret with write access to the project repo. Run C1, restart the session (Runtime > Restart session), then run C2.


In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.22", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through the queue (runs/queue_lucia_0.8b.json on the Hub; new runs can be added while this runs)
from qwen_symbolic_thinking.queue import run_queue

ran = run_queue("lucia_0.8b")
print("ran:", ran)
